# Module 01, Notebook 2: ITS Diagnostics

## Learning Objectives
By completing this notebook, you will:
1. Detect autocorrelation in ITS residuals and understand its consequences
2. Run posterior predictive checks to identify model misspecification
3. Conduct placebo tests to assess ITS validity
4. Use residual plots to diagnose systematic model failures
5. Interpret ArviZ diagnostic outputs for ITS models

## Why Diagnostics Matter

An ITS analysis produces a causal estimate only if the model is correctly specified. Diagnostics answer:
- **Is the pre-trend well-modeled?** (Residual structure, posterior predictive check)
- **Are errors autocorrelated?** (ACF of residuals, Durbin-Watson)
- **Would we find a fake effect at wrong dates?** (Placebo test)
- **Has the model converged?** (R-hat, ESS, trace plots)

A model that fails any of these diagnostics should not be reported without addressing the failure.

## Prerequisites
- Completed Notebook 01 (fitted smoking ban ITS model)
- Module 01 Guide 2 (segmented regression, autocorrelation)

## Estimated Time: 15 minutes

---

In [ ]:
learning_objectives(["Detect autocorrelation in ITS residuals and understand its consequences", "Run posterior predictive checks to identify model misspecification", "Conduct placebo tests to assess ITS validity", "Use residual plots to diagnose systematic model failures", "Interpret ArviZ diagnostic outputs for ITS models"])

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

In [ ]:
# Apply course styling
apply_course_theme()
apply_plot_theme()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import causalpy as cp
import arviz as az
from scipy import stats
from statsmodels.stats.stattools import durbin_watson
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import statsmodels.formula.api as smf

np.random.seed(42)
plt.style.use("seaborn-v0_8-whitegrid")
%matplotlib inline

print("Imports complete.")

In [ ]:
# Rebuild the smoking ban dataset from Notebook 01
# (same parameters — copy here for self-containment)

np.random.seed(42)
n_months = 48
intervention_month = 24

months = np.arange(n_months)
calendar_month = months % 12
seasonal_effect = (
    6.0 * np.cos(2 * np.pi * calendar_month / 12)
    + 2.0 * np.sin(2 * np.pi * calendar_month / 12)
)
treated = (months >= intervention_month).astype(float)
t_post = np.maximum(months - intervention_month, 0).astype(float)

ami_rate = (
    85.0
    - 0.15 * months
    + (-12.0) * treated
    + (-0.10) * t_post * treated
    + seasonal_effect
    + np.random.normal(0, 4.0, n_months)
)

df = pd.DataFrame({
    "month": months,
    "ami_rate": ami_rate,
    "treated": treated,
    "t_post": t_post,
    "calendar_month": calendar_month,
})

# Fit the seasonally adjusted model
its_model = cp.pymc_experiments.InterruptedTimeSeries(
    data=df,
    treatment_time=intervention_month,
    formula="ami_rate ~ 1 + month + treated + t_post + C(calendar_month)",
    model=cp.pymc_models.LinearRegression(
        sample_kwargs={
            "draws": 1000,
            "tune": 1000,
            "chains": 4,
            "progressbar": True,
            "random_seed": 42,
        }
    ),
)

print("Data and model ready.")

## 1. Residual Analysis

Residual plots reveal systematic patterns the model failed to capture. For ITS, we look for:
- No systematic trend in residuals (good pre-trend fit)
- No seasonal patterns in residuals (seasonality captured)
- No structural breaks in the residuals
- No dramatic outliers

In [ ]:
# Compute model residuals using OLS for visual analysis
# (OLS gives identical point estimates to Bayesian with flat priors)

# Fit OLS version for residual extraction
ols_fit = smf.ols(
    "ami_rate ~ 1 + month + treated + t_post + C(calendar_month)",
    data=df,
).fit()

residuals = ols_fit.resid
fitted_values = ols_fit.fittedvalues

fig, axes = plt.subplots(2, 2, figsize=(13, 8))

# Panel 1: Residuals over time
ax = axes[0, 0]
ax.plot(df["month"], residuals, "o-", color="#3498db", linewidth=1, markersize=3)
ax.axhline(0, color="red", linestyle="--", linewidth=1.5)
ax.axvline(intervention_month, color="#27ae60", linestyle="--", linewidth=2,
           label="Intervention")
ax.set_xlabel("Month")
ax.set_ylabel("Residual")
ax.set_title("Residuals Over Time")
ax.legend()

# Panel 2: Residuals vs Fitted
ax = axes[0, 1]
ax.scatter(fitted_values, residuals, alpha=0.6, color="#e74c3c", s=25)
ax.axhline(0, color="red", linestyle="--", linewidth=1.5)
ax.set_xlabel("Fitted Values")
ax.set_ylabel("Residual")
ax.set_title("Residuals vs Fitted")

# Panel 3: Q-Q plot (normality check)
ax = axes[1, 0]
stats.probplot(residuals, dist="norm", plot=ax)
ax.set_title("Normal Q-Q Plot")

# Panel 4: Histogram of residuals
ax = axes[1, 1]
ax.hist(residuals, bins=20, edgecolor="white", color="#9b59b6", alpha=0.8)
x_range = np.linspace(residuals.min(), residuals.max(), 100)
ax.plot(x_range,
        stats.norm.pdf(x_range, residuals.mean(), residuals.std()) * len(residuals) * (residuals.max() - residuals.min()) / 20,
        "r-", linewidth=2, label="Normal fit")
ax.set_xlabel("Residual")
ax.set_ylabel("Count")
ax.set_title("Residual Distribution")
ax.legend()

plt.suptitle("ITS Residual Diagnostics", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

print("What to look for:")
print("  - Residuals over time: No systematic trend (good)")
print("  - Residuals vs Fitted: Random scatter (good)")
print("  - Q-Q plot: Points near the diagonal (normality holds)")
print("  - Histogram: Approximately bell-shaped")

## 2. Autocorrelation Diagnostics

In [ ]:
# Autocorrelation and Partial Autocorrelation Function plots
# ACF: shows correlation at each lag
# PACF: shows the unique contribution at each lag after removing lower-order effects

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

plot_acf(residuals, lags=20, ax=axes[0], alpha=0.05)
axes[0].set_title("Autocorrelation Function (ACF) of Residuals", fontsize=12)
axes[0].set_xlabel("Lag (months)")
axes[0].set_ylabel("Autocorrelation")

plot_pacf(residuals, lags=20, ax=axes[1], alpha=0.05, method="ywm")
axes[1].set_title("Partial Autocorrelation Function (PACF) of Residuals", fontsize=12)
axes[1].set_xlabel("Lag (months)")
axes[1].set_ylabel("Partial Autocorrelation")

plt.tight_layout()
plt.show()

# Durbin-Watson test
dw_stat = durbin_watson(residuals)
rho_estimate = np.corrcoef(residuals[:-1], residuals[1:])[0, 1]

print("Autocorrelation Assessment")
print("=" * 45)
print(f"  Durbin-Watson statistic: {dw_stat:.3f}")
print(f"  Estimated lag-1 autocorrelation: {rho_estimate:.3f}")
print()
if dw_stat < 1.5:
    print("  RESULT: Positive autocorrelation detected.")
    print("  Action: Consider Newey-West standard errors or AR(1) error model.")
elif dw_stat > 2.5:
    print("  RESULT: Negative autocorrelation detected.")
else:
    print("  RESULT: No significant autocorrelation (DW ≈ 2).")
    print("  The standard error estimates are reliable.")

## 3. Posterior Predictive Check

The posterior predictive check (PPC) generates synthetic datasets from the fitted posterior and compares them to the observed data. If the model is well-specified, the observed data should look like a typical draw from the posterior predictive distribution.

In [ ]:
# Posterior Predictive Check
# Generate data from the posterior and overlay on observed data

import pymc as pm

# Sample posterior predictive
with its_model.model:
    ppc = pm.sample_posterior_predictive(its_model.idata, random_seed=42)

# Extract posterior predictive samples
y_ppc = ppc.posterior_predictive["y_hat"].values  # Shape: (chains, draws, n_obs)
y_ppc_flat = y_ppc.reshape(-1, n_months)  # Shape: (total_samples, n_obs)

fig, ax = plt.subplots(figsize=(13, 5))

# Plot posterior predictive samples (thin, transparent)
for i in range(0, min(100, y_ppc_flat.shape[0]), 1):
    ax.plot(df["month"], y_ppc_flat[i], color="#3498db", alpha=0.05, linewidth=0.5)

# Plot posterior predictive mean
ppc_mean = y_ppc_flat.mean(axis=0)
ax.plot(df["month"], ppc_mean, color="#2980b9", linewidth=2, label="PPC mean")

# Plot observed data
ax.scatter(df["month"], df["ami_rate"], color="#e74c3c", s=20, zorder=5, label="Observed")
ax.axvline(intervention_month, color="#27ae60", linestyle="--", linewidth=2, label="Intervention")

ax.set_xlabel("Month", fontsize=12)
ax.set_ylabel("AMI Rate per 100,000", fontsize=12)
ax.set_title("Posterior Predictive Check: Model vs Observed Data", fontsize=13)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

print("\nPPC interpretation:")
print("  Blue band = range of data generated from the fitted model")
print("  Red dots = actual observed data")
print("  If dots are outside the blue band, the model is misspecified")

In [ ]:
# Statistical PPC summary
# Compare observed statistics to their posterior predictive distributions

observed_stats = {
    "Mean AMI rate": df["ami_rate"].mean(),
    "Std AMI rate": df["ami_rate"].std(),
    "Min AMI rate": df["ami_rate"].min(),
    "Max AMI rate": df["ami_rate"].max(),
}

ppc_stats = {
    "Mean AMI rate": y_ppc_flat.mean(axis=1),
    "Std AMI rate": y_ppc_flat.std(axis=1),
    "Min AMI rate": y_ppc_flat.min(axis=1),
    "Max AMI rate": y_ppc_flat.max(axis=1),
}

print("Posterior Predictive Summary Statistics")
print("=" * 60)
print(f"{'Statistic':<20} {'Observed':>10} {'PPC Mean':>10} {'P(>obs)':>10}")
print("-" * 60)

for stat_name in observed_stats:
    obs_val = observed_stats[stat_name]
    ppc_dist = ppc_stats[stat_name]
    ppc_mean = ppc_dist.mean()
    p_exceed = (ppc_dist > obs_val).mean()
    print(f"  {stat_name:<18} {obs_val:>10.2f} {ppc_mean:>10.2f} {p_exceed:>10.2f}")

print()
print("Values near 0.5 indicate good model calibration.")
print("Values near 0 or 1 suggest the model misses an important pattern.")

## 4. Placebo Test

A placebo test runs the ITS at a fake intervention date in the pre-period. If the design is valid, we should find no significant effect at the fake date. A significant effect at the placebo date suggests:
- The pre-period trend is non-linear or unstable
- Something else was changing during the pre-period
- The model is misspecified

In [ ]:
# Placebo Test: Run ITS at fake intervention dates in the pre-period
# For each placebo date, use only the pre-intervention data

pre_df = df[df["treated"] == 0].copy()
pre_months = pre_df["month"].values

# Test placebo interventions at several dates in the pre-period
# Exclude the first and last few months to have enough pre and post in the placebo period
placebo_dates = [6, 9, 12, 15, 18]

placebo_results = []

for placebo_t in placebo_dates:
    # Build placebo dataset using only pre-intervention data
    placebo_df = pre_df.copy()
    placebo_df["placebo_treated"] = (placebo_df["month"] >= placebo_t).astype(float)
    placebo_df["placebo_t_post"] = np.maximum(
        placebo_df["month"] - placebo_t, 0
    ).astype(float)

    # Fit OLS (faster than Bayesian for placebo grid search)
    placebo_fit = smf.ols(
        "ami_rate ~ 1 + month + placebo_treated + placebo_t_post + C(calendar_month)",
        data=placebo_df,
    ).fit()

    placebo_results.append({
        "placebo_date": placebo_t,
        "level_change": placebo_fit.params["placebo_treated"],
        "level_change_se": placebo_fit.bse["placebo_treated"],
        "level_change_pval": placebo_fit.pvalues["placebo_treated"],
    })

placebo_df_results = pd.DataFrame(placebo_results)

# Real model result
real_level_change = its_model.idata.posterior["treated"].values.flatten().mean()

# Plot placebo results
fig, ax = plt.subplots(figsize=(10, 5))

ax.errorbar(
    placebo_df_results["placebo_date"],
    placebo_df_results["level_change"],
    yerr=1.96 * placebo_df_results["level_change_se"],
    fmt="o",
    color="#95a5a6",
    capsize=5,
    markersize=8,
    label="Placebo estimates (95% CI)",
)

# Real intervention estimate
ax.scatter([intervention_month], [real_level_change], color="#e74c3c",
           s=150, zorder=5, label=f"Real intervention estimate: {real_level_change:.1f}")

ax.axhline(0, color="black", linestyle="--", linewidth=1.5)
ax.axvline(intervention_month, color="#27ae60", linestyle=":", linewidth=2)

ax.set_xlabel("Intervention Date (Month Index)", fontsize=12)
ax.set_ylabel("Estimated Level Change", fontsize=12)
ax.set_title("Placebo Test: Level Change Estimates at Fake Intervention Dates", fontsize=13)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

print("Placebo Test Results:")
print(placebo_df_results.to_string(index=False))
print()
print("Interpretation:")
print("  Placebo estimates should be near zero with wide CIs (no spurious effect)")
print("  The real intervention estimate should stand out as clearly different")

## 5. Complete Diagnostic Report

Summarize all diagnostics in a structured report format.

In [ ]:
# Generate a structured diagnostic report

# Convergence
rhat_max = float(az.rhat(its_model.idata).to_array().max())
ess_min = float(az.ess(its_model.idata).to_array().min())
n_div = int(its_model.idata.sample_stats["diverging"].sum().item())

# Autocorrelation
dw = durbin_watson(residuals)
rho = np.corrcoef(residuals[:-1], residuals[1:])[0, 1]

# PPC - fraction of observed within 95% PPC interval
ppc_lower = np.percentile(y_ppc_flat, 2.5, axis=0)
ppc_upper = np.percentile(y_ppc_flat, 97.5, axis=0)
within_95 = ((df["ami_rate"].values >= ppc_lower) & (df["ami_rate"].values <= ppc_upper)).mean()

# Placebo
max_placebo_pval = placebo_df_results["level_change_pval"].min()  # Most significant placebo

print("ITS DIAGNOSTIC REPORT")
print("=" * 55)
print()
print("1. MCMC CONVERGENCE")
print(f"   Max R-hat:              {rhat_max:.4f}  {'PASS' if rhat_max < 1.01 else 'FAIL'}")
print(f"   Min Effective Samples:  {ess_min:.0f}      {'PASS' if ess_min > 400 else 'FAIL'}")
print(f"   Divergences:            {n_div}         {'PASS' if n_div < 5 else 'FAIL'}")
print()
print("2. AUTOCORRELATION")
print(f"   Durbin-Watson:          {dw:.3f}  {'PASS (≈2)' if 1.5 < dw < 2.5 else 'WARNING'}")
print(f"   Lag-1 autocorrelation:  {rho:.3f}")
print()
print("3. POSTERIOR PREDICTIVE CHECK")
print(f"   Obs within 95% PPC:     {within_95:.1%}  {'PASS (should be ≈95%)' if 0.85 <= within_95 <= 0.99 else 'WARNING'}")
print()
print("4. PLACEBO TEST")
print(f"   Min placebo p-value:    {max_placebo_pval:.3f}  {'PASS (>0.05)' if max_placebo_pval > 0.05 else 'WARNING'}")
print()
print("=" * 55)
print()
all_pass = (rhat_max < 1.01) and (ess_min > 400) and (n_div < 5) and (0.85 <= within_95 <= 0.99) and (max_placebo_pval > 0.05)
print(f"OVERALL STATUS: {'PASS — proceed with causal interpretation' if all_pass else 'ISSUES DETECTED — review before reporting'}")

## Summary

### Diagnostic Checklist for ITS

| Diagnostic | Tool | Pass Criterion |
|-----------|------|---------------|
| MCMC convergence | R-hat, ESS, trace plots | R-hat < 1.01, ESS > 400 |
| Autocorrelation | DW test, ACF/PACF plots | DW ≈ 2, no significant ACF spikes |
| Model fit | Posterior predictive check | ~95% of obs within 95% PPC band |
| Design validity | Placebo test | No significant effects at fake dates |
| Residual normality | Q-Q plot | Points near diagonal |

### What to Do When Diagnostics Fail

1. **R-hat > 1.01:** Increase `tune`, increase `target_accept`, check for model misspecification
2. **High autocorrelation:** Add AR(1) error term or use Newey-West SE correction
3. **PPC mismatch:** Model is missing important features — add seasonality, non-linear trend, or outlier handling
4. **Significant placebo effect:** Pre-trend is unstable — investigate and model any pre-intervention structural changes

### What's Next
**Notebook 3:** Handling seasonality in ITS — Fourier terms, month dummies, STL decomposition

In [ ]:
key_takeaways([
    "Core concept from this notebook demonstrated with working code",
    "Key parameters and their effects on results",
    "When to apply this technique vs alternatives"
])